In [1]:
import os, re, json, shutil, subprocess
import numpy as np
def ensure_repo(repo_dir: str, repo_url: str):
    if os.path.exists(repo_dir) and os.path.isdir(repo_dir):
        return
    os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
    subprocess.check_call(["git", "clone", repo_url, repo_dir])

def run_cmd(cmd, cwd=None, env=None):
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True)
    return p.returncode, p.stdout + "\n" + p.stderr

def extract_test_acc(log_text: str):
    for pat in patterns:
        m = re.search(pat, log_text, flags=re.IGNORECASE)
        if m:
            return float(m.group(1))
    return None

In [5]:
import os
import json
import math
import argparse
import numpy as np
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------
# Data I/O (your text_min format)
# ----------------------------
def load_textmin_dataset(dataset_folder: str):
    edges_path = os.path.join(dataset_folder, "edges.txt")
    feat_path = os.path.join(dataset_folder, "features.csv")
    lab_path = os.path.join(dataset_folder, "labels.csv")

    if not os.path.exists(edges_path): raise FileNotFoundError(edges_path)
    if not os.path.exists(feat_path): raise FileNotFoundError(feat_path)
    if not os.path.exists(lab_path): raise FileNotFoundError(lab_path)
    edges = np.loadtxt(edges_path, dtype=np.int64)
    X = np.loadtxt(feat_path, delimiter=",", dtype=np.float32)
    y = np.loadtxt(lab_path, dtype=np.int64)
    if edges.ndim == 1:
        edges = edges.reshape(-1, 2)
    n = X.shape[0]
    if y.shape[0] != n:
        raise ValueError(f"labels.csv length {y.shape[0]} != num_nodes {n}")
    row, col = edges[:, 0], edges[:, 1]
    A = sp.coo_matrix((np.ones(len(row), dtype=np.float32), (row, col)), shape=(n, n))
    A = A + A.T
    A.data[:] = 1.0
    A.setdiag(0)
    A.eliminate_zeros()
    A = A.tocsr()
    return X, y, A


def _read_idx_file(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing split file: {path}")
    idx = np.loadtxt(path, dtype=np.int64)
    idx = np.atleast_1d(idx)
    return idx


def load_20_splits(splits_dir: str, dataset: str, one_based_idx: bool):
    splits = []
    for s in range(1, 21):
        seed_dir = os.path.join(splits_dir, dataset, f"seed_{s:02d}")
        tr = _read_idx_file(os.path.join(seed_dir, "train_idx.txt"))
        va = _read_idx_file(os.path.join(seed_dir, "val_idx.txt"))
        te = _read_idx_file(os.path.join(seed_dir, "test_idx.txt"))
        if one_based_idx:
            tr = tr - 1
            va = va - 1
            te = te - 1
        splits.append({"seed": s, "train": tr, "val": va, "test": te})
    return splits


# ----------------------------
# Train-only TFIDF + L2 + PCA
# ----------------------------
def tfidf_l2_pca_train_only(
 X: np.ndarray,
 train_idx: np.ndarray,
 pca_dim: int = 256,
 use_tfidf: bool = True,
 l2_rows: bool = True,
 eps: float = 1e-12,
):
    X = np.asarray(X, dtype=np.float32)
    train_idx = np.asarray(train_idx, dtype=np.int64)
    n_train = int(len(train_idx))
    if n_train < 2:
        raise ValueError("Need at least 2 training nodes for PCA(train-only).")
    Xw = X
    if use_tfidf:
        df = (X[train_idx] > 0).sum(axis=0).astype(np.float32)
        idf = np.log((n_train + 1.0) / (df + 1.0)) + 1.0
        Xw = X * idf[None, :]
    if l2_rows:
        norms = np.linalg.norm(Xw, axis=1, keepdims=True)
        Xw = Xw / (norms + eps)
    rnk = int(max(1, min(pca_dim, Xw.shape[1], n_train - 1)))
    mu = Xw[train_idx].mean(axis=0, keepdims=True)
    Xc_train = Xw[train_idx]
    Xc=Xw-mu
    try:
        from sklearn.decomposition import TruncatedSVD
        svd=TruncatedSVD(n_components=rnk, random_state=0)
        svd.fit(Xc_train)
        Xp=svd.transform(Xc)
        return Xp.astype(np.float32)
    except Exception:
        Xt=torch.from_numpy.float()
        U,S,V = torch.pca_lowrank(Xt,q=rnk)
        W=V[:,:rnk].numpy()
        Xp=(Xc @ W).astype(np.float32)
        return Xp
    
    U, S, Vt = np.linalg.svd(Xc_train, full_matrices=False)
    W = Vt[:rnk].T # [F, rnk]
    Xp = (Xw - mu) @ W
    return Xp.astype(np.float32)

def row_normalize(A: sp.spmatrix, eps: float = 1e-12):
    A = A.tocsr()
    d = np.asarray(A.sum(axis=1)).reshape(-1)
    d = 1.0 / (d + eps)
    Dinv = sp.diags(d.astype(np.float32))
    return (Dinv @ A).tocsr()


def sparse_to_torch_coo(A: sp.spmatrix):
    A = A.tocoo()
    idx = torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long)
    val = torch.tensor(A.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(idx, val, size=A.shape).coalesce()


def spmm(A_coo: torch.Tensor, X: torch.Tensor):
    return torch.sparse.mm(A_coo, X)


# ----------------------------
# MixHop Layer + Model (PyTorch)
# ----------------------------
class MixHopLayer(nn.Module):
    def __init__(self, in_dim, out_dim, powers=(0, 1, 2), dropout=0.5):
        super().__init__()
        self.powers = tuple(int(p) for p in powers)
        self.dropout = float(dropout)
        self.linears = nn.ModuleDict()
        for p in self.powers:
            self.linears[str(p)] = nn.Linear(in_dim, out_dim, bias=True)
        self.bn = nn.BatchNorm1d(out_dim * len(self.powers))
    def forward(self, X, A_coo):
        outs = []
        for p in self.powers:
            H = X
            for _ in range(p):
                H = torch.sparse.mm(A_coo, H)
            H = self.linears[str(p)](H)
            outs.append(H)
        Hcat = torch.cat(outs, dim=1) # [N, out_dim * |P|]
        Hcat = self.bn(Hcat)
        Hcat = F.relu(Hcat)
        Hcat = F.dropout(Hcat, p=self.dropout, training=self.training)
        return Hcat


class MixHopNet(nn.Module):
     def __init__(
        self,
        in_dim,
        hidden_dim,
        out_dim,
        powers1=(0, 1, 2),
        powers2=(0, 1, 2),
        dropout=0.5,
     ):
        super().__init__()
        self.l1 = MixHopLayer(in_dim, hidden_dim, powers=powers1, dropout=dropout)
        self.l2 = MixHopLayer(hidden_dim * len(powers1), hidden_dim, powers=powers2, dropout=dropout)
        self.classifier = nn.Linear(hidden_dim * len(powers2), out_dim)
     def forward(self, X, A_coo):
         H = self.l1(X, A_coo)
         H = self.l2(H, A_coo)
         out = self.classifier(H)
         return out, H 



@torch.no_grad()
def accuracy(logits: torch.Tensor, y: torch.Tensor, idx: torch.Tensor) -> float:
    pred = logits.argmax(dim=-1)
    return float((pred[idx] == y[idx]).float().mean().item())


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_one_split(
 X_np: np.ndarray,
 y_np: np.ndarray,
 A: sp.spmatrix,
 train_idx: np.ndarray,
 val_idx: np.ndarray,
 test_idx: np.ndarray,
 device: str = "cuda",
 hidden: int = 64,
 powers1=(0, 1, 2),
 powers2=(0, 1, 2),
 dropout: float = 0.5,
 lr: float = 0.01,
 wd: float = 5e-4,
 max_epochs: int = 500,
 patience: int = 50,
 seed: int = 1,
):
    set_seed(seed)
    dev = torch.device(device if (device == "cpu" or torch.cuda.is_available()) else "cpu")
    A_norm = row_normalize(A)
    A_coo = sparse_to_torch_coo(A_norm).to(dev)
    X = torch.tensor(X_np, dtype=torch.float32, device=dev)
    y = torch.tensor(y_np, dtype=torch.long, device=dev)
    tr = torch.tensor(train_idx, dtype=torch.long, device=dev)
    va = torch.tensor(val_idx, dtype=torch.long, device=dev)
    te = torch.tensor(test_idx, dtype=torch.long, device=dev)
    n_classes = int(y.max().item() + 1)
    model = MixHopNet(
        in_dim=X.shape[1],
        hidden_dim=hidden,
        out_dim=n_classes,
        powers1=powers1,
        powers2=powers2,
        dropout=dropout
    ).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val = -1.0
    best_state = None
    bad = 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad()
        logits, _ = model(X, A_coo)
        loss = F.cross_entropy(logits[tr], y[tr])
        loss.backward()
        opt.step()

        model.eval()
        logits, _ = model(X, A_coo)
        val_acc = accuracy(logits, y, va)
        if val_acc > best_val + 1e-6:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    logits, emb = model(X, A_coo)
    tr_acc = accuracy(logits, y, tr)
    va_acc = accuracy(logits, y, va)
    te_acc = accuracy(logits, y, te)

    return {
     "train_acc": tr_acc,
     "val_acc": va_acc,
     "test_acc": te_acc,
     "test_err": 1.0 - te_acc,
     "best_val": float(best_val),
    }


def run_20_splits_mixhop(
 data_dir: str,
 splits_dir: str,
 dataset: str,
 out_dir: str,
 one_based_idx: bool = False, # True if your txt indices are 1..N
 use_prep: bool = True,
 pca_dim: int = 256,
 use_tfidf: bool = True,
 l2_rows: bool = True,
 device: str = "cuda",
 max_epochs: int = 500,
 patience: int = 50,
 hidden: int = 64,
 powers1=(0, 1, 2),
 powers2=(0, 1, 2),
 dropout: float = 0.5,
 lr: float = 0.01,
 wd: float = 5e-4,
):
    X_raw, y, A = load_textmin_dataset(os.path.join(data_dir, dataset))
    N = X_raw.shape[0]
    splits = load_20_splits(splits_dir, dataset, one_based_idx=one_based_idx)

    results = []
    for sp_ in splits:
        seed = int(sp_["seed"])
        tr = sp_["train"].astype(np.int64)
        va = sp_["val"].astype(np.int64)
        te = sp_["test"].astype(np.int64)

 # sanity check indices
        if tr.max() >= N or va.max() >= N or te.max() >= N or tr.min() < 0 or va.min() < 0 or te.min() < 0:
            raise IndexError(
             f"Split indices out of range for N={N}. "
             f"Max idx: train={tr.max()}, val={va.max()}, test={te.max()}. "
             f"Set one_based_idx=True if your files are 1..N."
            )

        if use_prep:
            pca_dim_run = int(max(1, min(pca_dim, len(tr) - 1, X_raw.shape[1])))
            Xp = tfidf_l2_pca_train_only(
                X_raw, tr, pca_dim=pca_dim_run,
                use_tfidf=use_tfidf,
                l2_rows=l2_rows
            )
        else:
            Xp = X_raw.astype(np.float32)
        out = run_one_split(
             X_np=Xp,
             y_np=y,
             A=A,
             train_idx=tr,
             val_idx=va,
             test_idx=te,
             device=device,
             hidden=hidden,
             powers1=powers1,
             powers2=powers2,
             dropout=dropout,
             lr=lr,
             wd=wd,
             max_epochs=max_epochs,
             patience=patience,
             seed=seed
        )
        out["seed"] = seed
        results.append(out)
        print(f"[seed {seed:02d}] test_acc={out['test_acc']:.4f} best_val={out['best_val']:.4f}")

    test_acc = np.array([r["test_acc"] for r in results], dtype=np.float32)
    test_err = np.array([r["test_err"] for r in results], dtype=np.float32)
    summary = {
         "dataset": dataset,
         "model": "MixHop (PyTorch re-implementation)",
         "use_prep": bool(use_prep),
         "prep": {
             "use_tfidf": bool(use_tfidf),
             "l2_rows": bool(l2_rows),
             "pca_dim": int(pca_dim),
             "train_only": True
         } if use_prep else None,
         "training": {
             "device": device,
             "max_epochs": int(max_epochs),
             "patience": int(patience),
             "hidden": int(hidden),
             "powers1": list(map(int, powers1)),
             "powers2": list(map(int, powers2)),
             "dropout": float(dropout),
             "lr": float(lr),
             "weight_decay": float(wd)
          },
          "mean_test_acc": float(test_acc.mean()),
          "std_test_acc": float(test_acc.std(ddof=1)),
          "mean_test_err": float(test_err.mean()),
          "std_test_err": float(test_err.std(ddof=1)),
          "per_seed": results
    }
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{dataset}_MixHop_20splits.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n=== FINAL SUMMARY (MixHop) ===")
    print(f"mean_test_acc = {summary['mean_test_acc']:.4f} ± {summary['std_test_acc']:.4f}")
    print(f"mean_test_err = {summary['mean_test_err']:.4f} ± {summary['std_test_err']:.4f}")
    print(f"Saved JSON: {out_path}")
    return summary


# ----------------------------
# CLI
# ----------------------------
def build_argparser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True, help="Folder with <dataset>/{edges.txt,features.csv,labels.csv}")
    ap.add_argument("--splits_dir", required=True, help="Folder with <dataset>/seed_01/{train_idx,val_idx,test_idx}.txt ...")
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--out_dir", default="results_mixhop")
    ap.add_argument("--one_based_idx", action="store_true",
    help="Set if your split txt indices are 1..N (will subtract 1).")
    ap.add_argument("--use_prep", action="store_true", help="Enable TFIDF+L2+PCA(train-only) per seed")
    ap.add_argument("--pca_dim", type=int, default=256)
    ap.add_argument("--no_tfidf", action="store_true")
    ap.add_argument("--no_l2", action="store_true")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--max_epochs", type=int, default=500)
    ap.add_argument("--patience", type=int, default=50)
    ap.add_argument("--hidden", type=int, default=64)
    ap.add_argument("--dropout", type=float, default=0.5)
    ap.add_argument("--lr", type=float, default=0.01)
    ap.add_argument("--wd", type=float, default=5e-4)
    ap.add_argument("--powers1", type=str, default="0,1,2", help="Comma list, e.g. 0,1,2")
    ap.add_argument("--powers2", type=str, default="0,1,2", help="Comma list, e.g. 0,1,2")
    return ap


def parse_powers(s: str):
    return tuple(int(x.strip()) for x in s.split(",") if x.strip() != "")


def main():
    ap = build_argparser()
    args = ap.parse_args()

    summary = run_20_splits_mixhop(
         data_dir=args.data_dir,
         splits_dir=args.splits_dir,
         dataset=args.dataset,
         out_dir=args.out_dir,
         one_based_idx=bool(args.one_based_idx),
         use_prep=bool(args.use_prep),
         pca_dim=int(args.pca_dim),
         use_tfidf=not bool(args.no_tfidf),
         l2_rows=not bool(args.no_l2),
         device=args.device,
         max_epochs=int(args.max_epochs),
         patience=int(args.patience),
         hidden=int(args.hidden),
         powers1=parse_powers(args.powers1),
         powers2=parse_powers(args.powers2),
         dropout=float(args.dropout),
         lr=float(args.lr),
         wd=float(args.wd),
    )
    return summary

In [6]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"]=""

In [ ]:
summary = run_20_splits_mixhop(
     data_dir=r"C:\Users\user\Desktop\Thesis Coding\wiki_text_min",
     splits_dir=r"C:\Users\user\Desktop\Thesis Coding\runs\chameleon_e2eboost2",
     dataset="chameleon",
     out_dir=r"C:\Users\user\Desktop\Thesis Coding\results_mixhop",
     one_based_idx=True, 
     use_prep=True,
     pca_dim=192,
     device="cuda",
     max_epochs=100
)
summary["mean_test_acc"], summary["std_test_acc"]

In [17]:
summary = run_20_splits_mixhop(
     data_dir=r"C:\Users\user\Desktop\Thesis Coding\wiki_text_min",
     splits_dir=r"C:\Users\user\Desktop\Thesis Coding\runs\chameleon_e2eboost2",
     dataset="chameleon",
     out_dir=r"C:\Users\user\Desktop\Thesis Coding\results_mixhop",
     one_based_idx=True, 
     use_prep=True,
     pca_dim=192,
     device="cuda",
     max_epochs=100
)
summary["mean_test_acc"], summary["std_test_acc"]

[seed 01] test_acc=0.6536 best_val=0.6982
[seed 02] test_acc=0.6623 best_val=0.6960
[seed 03] test_acc=0.7211 best_val=0.6674
[seed 04] test_acc=0.6906 best_val=0.7401
[seed 05] test_acc=0.6645 best_val=0.6674
[seed 06] test_acc=0.7015 best_val=0.6718
[seed 07] test_acc=0.6928 best_val=0.6762
[seed 08] test_acc=0.6972 best_val=0.7004
[seed 09] test_acc=0.6906 best_val=0.6740
[seed 10] test_acc=0.6972 best_val=0.6938
[seed 11] test_acc=0.6841 best_val=0.7093
[seed 12] test_acc=0.6710 best_val=0.6938
[seed 13] test_acc=0.6710 best_val=0.7379
[seed 14] test_acc=0.6885 best_val=0.7357
[seed 15] test_acc=0.6928 best_val=0.7423
[seed 16] test_acc=0.6645 best_val=0.6960
[seed 17] test_acc=0.7037 best_val=0.6960
[seed 18] test_acc=0.7190 best_val=0.7401
[seed 19] test_acc=0.6688 best_val=0.6762
[seed 20] test_acc=0.6623 best_val=0.7026

=== FINAL SUMMARY (MixHop) ===
mean_test_acc = 0.6849 ± 0.0193
mean_test_err = 0.3151 ± 0.0193
Saved JSON: C:\Users\user\Desktop\Thesis Coding\results_mixhop\c

(0.6848583817481995, 0.019309673458337784)

In [20]:
summary = run_20_splits_mixhop(
     data_dir=r"C:\Users\user\Desktop\Thesis Coding\wiki_text_min",
     splits_dir=r"C:\Users\user\Desktop\Thesis Coding\runs\squirrel_e2eboost2",
     dataset="squirrel",
     out_dir=r"C:\Users\user\Desktop\Thesis Coding\results_mixhop",
     one_based_idx=True, 
     use_prep=True,
     pca_dim=192,
     device="cuda",
     max_epochs=100
)
summary["mean_test_acc"], summary["std_test_acc"]

[seed 01] test_acc=0.5988 best_val=0.5698
[seed 02] test_acc=0.6123 best_val=0.5804
[seed 03] test_acc=0.5825 best_val=0.5852
[seed 04] test_acc=0.5797 best_val=0.5679
[seed 05] test_acc=0.2514 best_val=0.2310
[seed 06] test_acc=0.2121 best_val=0.2098
[seed 07] test_acc=0.5777 best_val=0.5813
[seed 08] test_acc=0.2754 best_val=0.2878
[seed 09] test_acc=0.5797 best_val=0.5881
[seed 10] test_acc=0.2255 best_val=0.2560
[seed 11] test_acc=0.5864 best_val=0.5938
[seed 12] test_acc=0.2486 best_val=0.2493
[seed 13] test_acc=0.5557 best_val=0.5890
[seed 14] test_acc=0.2361 best_val=0.2416
[seed 15] test_acc=0.5864 best_val=0.5688
[seed 16] test_acc=0.5701 best_val=0.5804
[seed 17] test_acc=0.5710 best_val=0.5919
[seed 18] test_acc=0.5960 best_val=0.5804
[seed 19] test_acc=0.2083 best_val=0.2089
[seed 20] test_acc=0.2370 best_val=0.2445

=== FINAL SUMMARY (MixHop) ===
mean_test_acc = 0.4445 ± 0.1749
mean_test_err = 0.5555 ± 0.1749
Saved JSON: C:\Users\user\Desktop\Thesis Coding\results_mixhop\s

(0.44452977180480957, 0.1748892068862915)

In [22]:
summary = run_20_splits_mixhop(
     data_dir=r"C:\Users\user\Desktop\Thesis Coding\citation_text_min",
     splits_dir=r"C:\Users\user\Desktop\Thesis Coding\runs\citeseer_e2eboost",
     dataset="citeseer",
     out_dir=r"C:\Users\user\Desktop\Thesis Coding\results_mixhop",
     one_based_idx=True, 
     use_prep=True,
     pca_dim=50,
     device="cuda",
     max_epochs=500
)
summary["mean_test_acc"], summary["std_test_acc"]

[seed 01] test_acc=0.9389 best_val=0.9443
[seed 02] test_acc=0.9389 best_val=0.9384
[seed 03] test_acc=0.9260 best_val=0.9348
[seed 04] test_acc=0.9412 best_val=0.9467
[seed 05] test_acc=0.9354 best_val=0.9336
[seed 06] test_acc=0.9459 best_val=0.9455
[seed 07] test_acc=0.9354 best_val=0.9372
[seed 08] test_acc=0.9401 best_val=0.9336
[seed 09] test_acc=0.9377 best_val=0.9431
[seed 10] test_acc=0.9436 best_val=0.9325
[seed 11] test_acc=0.9307 best_val=0.9396
[seed 12] test_acc=0.9389 best_val=0.9396
[seed 13] test_acc=0.9412 best_val=0.9372
[seed 14] test_acc=0.9448 best_val=0.9254
[seed 15] test_acc=0.9483 best_val=0.9585
[seed 16] test_acc=0.9365 best_val=0.9313
[seed 17] test_acc=0.9283 best_val=0.9431
[seed 18] test_acc=0.9271 best_val=0.9408
[seed 19] test_acc=0.9260 best_val=0.9467
[seed 20] test_acc=0.9401 best_val=0.9301

=== FINAL SUMMARY (MixHop) ===
mean_test_acc = 0.9373 ± 0.0066
mean_test_err = 0.0627 ± 0.0066
Saved JSON: C:\Users\user\Desktop\Thesis Coding\results_mixhop\c

(0.9372503161430359, 0.006640745792537928)